# Part-of-Speech Tag Frequency Comparison

This notebook compares the part-of-speech (POS) tag frequencies between two documents:
- `PrideAndPrejudice.txt`
- `DFARS.txt`

using spaCy and matplotlib.

In [ ]:
import spacy
import matplotlib.pyplot as plt
import numpy as np
from collections import Counter

# Load spaCy English model
nlp = spacy.load('en_core_web_lg')

Let's open up the two documents. Each one is lengthy. **Pride and Prejudice** is a full-length novel, and the Defense Federal Acquisition Regulation System (**DFARS**) document is a lengthy excerpt from the full federal regulation.

In [ ]:
# Read the documents
with open('../data/PrideAndPrejudice.txt', encoding='utf-8') as f:
    pride_text = f.read()

with open('../data/DFARS.txt', encoding='utf-8') as f:
    dfars_text = f.read()

Next up, we will process both documents with spaCy. This will take a bit!

After we finish processing the documents, we'll generate a counter for the list of tokens in each document.

In [ ]:
# Process the documents with spaCy
pride_doc = nlp(pride_text)
dfars_doc = nlp(dfars_text)

# Count POS tags
pride_pos_counts = Counter([token.pos_ for token in pride_doc])
dfars_pos_counts = Counter([token.pos_ for token in dfars_doc])

Calculate part of speech tags. We'll generate the full set of part of speech tags across both documents. Then, we'll count how frequently each part of speech comes up in each document. After that, we'll use matplotlib to plot a simple column chart showing the number of uses of each part of speech in Pride and Prejudice vs DFARS.

Before we run this, what expectations do you have in comparing a 19th century novel to a 21st century bureaucratic document?

In [ ]:
# Get all POS tags present in either document
all_pos_tags = sorted(set(pride_pos_counts.keys()) | set(dfars_pos_counts.keys()))

# Prepare data for plotting
pride_freqs = [pride_pos_counts.get(tag, 0) for tag in all_pos_tags]
dfars_freqs = [dfars_pos_counts.get(tag, 0) for tag in all_pos_tags]

# Plot
x = range(len(all_pos_tags))
width = 0.35

plt.figure(figsize=(12,6))
plt.bar(x, pride_freqs, width, label='Pride and Prejudice')
plt.bar([i + width for i in x], dfars_freqs, width, label='DFARS')
plt.xlabel('POS Tag')
plt.ylabel('Frequency')
plt.title('POS Tag Frequency Comparison')
plt.xticks([i + width/2 for i in x], all_pos_tags, rotation=45)
plt.legend()
plt.tight_layout()
plt.show()

There are several measures of how difficult a particular document is to read. These include lexical diversity (the number of unique words in a document, divided by the total number of words in the document), average sentence length in characters, and Flesch-Kincaid grade level estimation.

In [ ]:
def calculate_lexical_diversity(doc):
    tokens = [token.text.lower() for token in doc if not token.is_punct]  # Remove punctuation
    unique_words = set(tokens)
    return len(unique_words) / len(tokens)

def calculate_average_sentence_length(doc):
    sentences = list(doc.sents)
    total_chars = sum(len(sent.text) for sent in sentences)  # Total characters
    num_sentences = len(sentences)
    return total_chars / num_sentences # Average sentence length

def estimate_flesch_kincaid_grade_level(doc):
    """Estimates Flesch-Kincaid grade level. Requires syllable count estimation."""
    num_words = len([token for token in doc if not token.is_punct]) # Count words, excluding punctuation
    num_sentences = len(list(doc.sents))

    # Syllable counting is approximate - replace with a more robust method if needed.
    def count_syllables(word):
        vowels = "aeiouy"
        count = 0
        last_was_vowel = False
        for char in word:
            if char.lower() in vowels:
                if not last_was_vowel:
                    count += 1
                last_was_vowel = True
            else:
                last_was_vowel = False
        if word.endswith("e"):
            count -= 1  # Remove silent 'e' at the end of words
        return max(1, count) # Ensure syllable count is at least 1

    total_syllables = sum(count_syllables(token.text.lower()) for token in doc if not token.is_punct and len(token.text)>0)

    # Flesch-Kincaid Grade Level Formula
    grade_level = 0.39 * (num_words / num_sentences) + 11.8 * (total_syllables / num_words) - 15.59
    return grade_level

Let's try each of these functions out on a simple string, just to see what kinds of results we might come up with.

In [ ]:
# Example Usage:
text = "This is a sample text to demonstrate difficulty estimation."
doc = nlp(text)
lexical_diversity = calculate_lexical_diversity(doc)
avg_sentence_length = calculate_average_sentence_length(doc)
grade_level = estimate_flesch_kincaid_grade_level(doc)

print(f"Lexical Diversity: {lexical_diversity}")
print(f"Average Sentence Length: {avg_sentence_length}")
print(f"Estimated Flesch-Kincaid Grade Level: {grade_level}")

Next up, we will perform this set of tests on Pride and Prejudice.

In [ ]:
lexical_diversity = calculate_lexical_diversity(pride_doc)
avg_sentence_length = calculate_average_sentence_length(pride_doc)
grade_level = estimate_flesch_kincaid_grade_level(pride_doc)

print(f"Lexical Diversity: {lexical_diversity}")
print(f"Average Sentence Length: {avg_sentence_length}")
print(f"Estimated Flesch-Kincaid Grade Level: {grade_level}")

Here is what it looks like for DFARS.

In [ ]:
lexical_diversity = calculate_lexical_diversity(dfars_doc)
avg_sentence_length = calculate_average_sentence_length(dfars_doc)
grade_level = estimate_flesch_kincaid_grade_level(dfars_doc)

print(f"Lexical Diversity: {lexical_diversity}")
print(f"Average Sentence Length: {avg_sentence_length}")
print(f"Estimated Flesch-Kincaid Grade Level: {grade_level}")

We can use a mathematical operation called **cosine similarity** to compare how similar two documents are. We won't dig into the math behind cosine similarity but will briefly touch upon it later in the talk.

In [ ]:
dfars_doc.similarity(pride_doc)

The highest value we can get for cosine similarity is 1.0, when there's a perfect match.

In [ ]:
dfars_doc.similarity(dfars_doc)

One tricky part of cosine similarity is that it is an aggregate measure. If we look at the first 2000 words of Pride and Prejudice and compare them to the first 2000 words of the DFARS, we see a different value for cosine similarity.

In [ ]:
pride_doc[:2000].similarity(dfars_doc[:2000])

And if we keep moving around in the documents, we won't get precisely the same value.

In [ ]:
pride_doc[8000:10000].similarity(dfars_doc[8000:10000])

In case you haven't read either of these documents, let's take a look at a representative sentence from each.

In [ ]:
pride_sent = list(pride_doc.sents)[1906]
print(pride_sent.text)

In [ ]:
dfars_sent = list(dfars_doc.sents)[1921]
print(dfars_sent.text)

How similar are these sentences?

In [ ]:
dfars_sent.similarity(pride_sent)

We would expect that the order of comparison shouldn't affect the results.

In [ ]:
pride_sent.similarity(dfars_sent)

And just to double-check that sentence behavior doesn't differ, we can compare a sentence to itself.

In [ ]:
dfars_sent.similarity(dfars_sent)

The way that spaCy performs comparison has to do with a concept known as **vectors**. Keep that term in mind as we'll come back to it later!

We can look for the most similar vectors to a given word.

In [ ]:
sample_word = "arrangement"

ms = nlp.vocab.vectors.most_similar(np.asarray([nlp.vocab.vectors[nlp.vocab.strings[sample_word]]]), n=10)
words = [nlp.vocab.strings[w] for w in ms[0][0]]
distances = ms[2][0]
res = [ f'{w} - {d}' for w, d in zip(words, distances)]
for r in res:
    print(r)

Note that "similar" might not be the same as "is a morpheme of" or "is a synonym of." These are typical relationships, but we can also see words that show up frequently near 'arrangement' like 'whereby' or 'provided.'